# LeetCode #96 — Unique Binary Search Trees: đếm BST và số Catalan

**Mức độ:** Medium · **Chủ đề:** Math, Dynamic Programming, Tree, Binary Search Tree, Binary Tree

Skeleton trên LeetCode:

```python
class Solution:
    def numTrees(self, n: int) -> int:
```

Bài #95 **liệt kê** mọi BST. Bài này chỉ cần **đếm** — và nhờ vậy làm được với `n` lớn hơn nhiều. Mục tiêu của notebook:

1. Nhận ra: số BST **chỉ phụ thuộc số nút**, không phụ thuộc giá trị cụ thể.
2. Công thức DP `G[n] = Σ G[i−1] · G[n−i]` — **quy tắc nhân** của tổ hợp.
3. Công thức đóng của **số Catalan** `C_n = C(2n, n) / (n + 1)`.
4. Nhận diện các bài khác cũng ra số Catalan.

# 1. Bài toán là gì?

Cho số nguyên `n`, trả về số **cây nhị phân tìm kiếm khác nhau về cấu trúc** có đúng `n` nút với các giá trị `1 … n`.

**Constraints (theo đề):** `1 <= n <= 19`

---

## Ví dụ

```text
n = 3  ->  5
n = 1  ->  1
```

(Năm cây với `n = 3` đã được vẽ ở notebook #95.)

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Cây nhị phân tìm kiếm (BST)
2. Quy hoạch động (Dynamic Programming — DP)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Cây nhị phân tìm kiếm (BST)

### BST là gì?

Cây nhị phân tìm kiếm (Binary Search Tree) là cây nhị phân thỏa, **với mọi nút**:

- mọi giá trị trong cây con **trái** `<` giá trị nút,
- mọi giá trị trong cây con **phải** `>` giá trị nút.

```text
        8
       / \
      3   10
     / \    \
    1   6    14
       / \   /
      4   7 13
```

Chú ý: điều kiện áp dụng cho **toàn bộ** cây con, không chỉ con trực tiếp (ví dụ `7` nằm bên trái `8` nên phải `< 8`).

### Ba tính chất quan trọng

1. **Tìm kiếm** giống tìm nhị phân: so với nút, nhỏ hơn thì sang trái, lớn hơn thì sang phải → `O(h)` (h = chiều cao).
2. **Duyệt inorder cho dãy tăng dần**: `1 3 4 6 7 8 10 13 14`.
3. Cây **cân bằng** có `h ≈ log₂ n`; cây lệch có `h = n` (tệ như danh sách).

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def bst_chen(root, x):
    if root is None:
        return TreeNode(x)
    if x < root.val:
        root.left = bst_chen(root.left, x)
    else:
        root.right = bst_chen(root.right, x)
    return root

def bst_tim(root, x):
    buoc = 0
    while root:
        buoc += 1
        if x == root.val:
            return True, buoc
        root = root.left if x < root.val else root.right
    return False, buoc

def bst_inorder(node, out):
    if node:
        bst_inorder(node.left, out); out.append(node.val); bst_inorder(node.right, out)

r = None
for x in [8, 3, 10, 1, 6, 14, 4, 7, 13]:
    r = bst_chen(r, x)
out = []; bst_inorder(r, out)
print("inorder:", out)
print("tìm 7:", bst_tim(r, 7), "| tìm 5:", bst_tim(r, 5), "(đã tìm, số bước)")

inorder: [1, 3, 4, 6, 7, 8, 10, 13, 14]
tìm 7: (True, 4) | tìm 5: (False, 4) (đã tìm, số bước)


### Dựng BST cân bằng từ dãy đã sắp xếp

Lấy phần tử **giữa** làm gốc, nửa trái dựng cây con trái, nửa phải dựng cây con phải (chia để trị). Chiều cao chỉ `O(log n)`.

In [2]:
def bst_tu_mang(a, lo=0, hi=None):
    if hi is None:
        hi = len(a) - 1
    if lo > hi:
        return None
    mid = (lo + hi) // 2
    return TreeNode(a[mid], bst_tu_mang(a, lo, mid - 1), bst_tu_mang(a, mid + 1, hi))

def cao(n):
    return 0 if n is None else 1 + max(cao(n.left), cao(n.right))

lech = None
for x in range(1, 16):
    lech = bst_chen(lech, x)                  # chèn tăng dần → cây lệch
print("15 số chèn tăng dần: cao", cao(lech), "| dựng từ giữa: cao", cao(bst_tu_mang(list(range(1, 16)))))

15 số chèn tăng dần: cao 15 | dựng từ giữa: cao 4


### Kiểm tra một cây có phải BST

Sai lầm phổ biến: chỉ so mỗi nút với **con trực tiếp**. Đúng: truyền xuống **khoảng cho phép** `(lo, hi)`; sang trái thì cận trên thành giá trị nút, sang phải thì cận dưới thành giá trị nút. (Hoặc: inorder phải tăng ngặt.)

In [3]:
def la_bst(node, lo=float("-inf"), hi=float("inf")):
    if node is None:
        return True
    if not (lo < node.val < hi):
        return False
    return la_bst(node.left, lo, node.val) and la_bst(node.right, node.val, hi)

def chi_so_con(node):                          # cách SAI
    if node is None:
        return True
    if node.left and node.left.val >= node.val: return False
    if node.right and node.right.val <= node.val: return False
    return chi_so_con(node.left) and chi_so_con(node.right)

bay = TreeNode(5, TreeNode(1), TreeNode(7, TreeNode(4), TreeNode(8)))   # 4 nằm bên phải 5 → không phải BST
print("đúng:", la_bst(bay), "| cách chỉ so con trực tiếp (sai):", chi_so_con(bay))

đúng: False | cách chỉ so con trực tiếp (sai): True


## 2.2 Quy hoạch động (Dynamic Programming — DP)

### Khi nào dùng DP?

Quy hoạch động dùng cho bài toán có hai đặc điểm:

1. **Cấu trúc con tối ưu**: đáp án của bài lớn được ghép từ đáp án của các **bài con** nhỏ hơn.
2. **Bài con chồng lấp**: cùng một bài con xuất hiện **nhiều lần** nếu đệ quy thẳng.

Thay vì tính lại, ta **lưu** đáp án mỗi bài con vào bảng và dùng lại → mỗi bài con tính đúng một lần.

### Bốn câu hỏi khi thiết kế DP

1. **Trạng thái**: `dp[i]` nghĩa là gì? (viết thành câu tiếng Việt rõ ràng)
2. **Chuyển trạng thái**: `dp[i]` tính từ những `dp[j]` nào?
3. **Cơ sở**: giá trị nhỏ nhất biết sẵn (`dp[0]`, `dp[1]`…).
4. **Đáp án**: là `dp[n]`, hay `max(dp)`, hay …?

### Ví dụ: leo cầu thang

Mỗi bước leo 1 hoặc 2 bậc. Có bao nhiêu cách leo `n` bậc?

- Trạng thái: `dp[i]` = số cách lên tới bậc `i`.
- Chuyển: bước cuối cùng là 1 bậc (từ `i−1`) hoặc 2 bậc (từ `i−2`) → `dp[i] = dp[i−1] + dp[i−2]`.
- Cơ sở: `dp[0] = 1` (đứng yên), `dp[1] = 1`.

In [4]:
from functools import lru_cache

# Cách 1: đệ quy thuần — chậm vì tính lặp
def leo_de_quy(n):
    return 1 if n <= 1 else leo_de_quy(n - 1) + leo_de_quy(n - 2)

# Cách 2: đệ quy + ghi nhớ (top-down, memoization)
@lru_cache(maxsize=None)
def leo_memo(n):
    return 1 if n <= 1 else leo_memo(n - 1) + leo_memo(n - 2)

# Cách 3: điền bảng từ nhỏ tới lớn (bottom-up, tabulation)
def leo_bang(n):
    dp = [0] * (n + 1)
    dp[0] = 1
    if n >= 1:
        dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n], dp

# Cách 4: chỉ giữ hai giá trị gần nhất — O(1) bộ nhớ
def leo_gon(n):
    a, b = 1, 1                 # dp[i-2], dp[i-1]
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

print("n = 10:", leo_de_quy(10), leo_memo(10), leo_bang(10)[0], leo_gon(10))
print("bảng dp:", leo_bang(10)[1])

n = 10: 89 89 89 89
bảng dp: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### Top-down hay bottom-up?

| | Top-down (memo) | Bottom-up (bảng) |
|---|---|---|
| Viết | gần với công thức đệ quy, dễ nghĩ | phải chọn **thứ tự** điền bảng |
| Chỉ tính bài con cần thiết | có | không (tính hết) |
| Rủi ro | độ sâu đệ quy | không |
| Tối ưu bộ nhớ | khó | dễ (giữ vài hàng/ô gần nhất) |

### Ví dụ thứ hai: chọn hay không chọn (House Robber)

Dãy nhà có tiền `a[i]`, không được lấy hai nhà **liền nhau**. Lấy tối đa bao nhiêu?

- `dp[i]` = số tiền tối đa khi chỉ xét `i` nhà đầu.
- Nhà thứ `i` (chỉ số `i−1`): **không lấy** → `dp[i−1]`; **lấy** → `dp[i−2] + a[i−1]`. Lấy `max`.

In [5]:
def ke_trom(a):
    dp = [0] * (len(a) + 1)
    if a:
        dp[1] = a[0]
    for i in range(2, len(a) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + a[i - 1])
    return dp[-1], dp

print("ke_trom([2, 7, 9, 3, 1]) =", ke_trom([2, 7, 9, 3, 1]))

ke_trom([2, 7, 9, 3, 1]) = (12, [0, 2, 7, 11, 11, 12])


### Lỗi hay gặp

- Định nghĩa trạng thái mơ hồ → công thức sai. Luôn viết `dp[i] = …` bằng lời trước.
- Lệch chỉ số giữa `dp` (dài `n+1`) và mảng đầu vào (dài `n`).
- Điền bảng sai thứ tự (dùng ô chưa tính).
- Quên cơ sở hoặc khởi tạo sai (`0` thay vì `-∞` cho bài tìm max có giá trị âm…).
- Tối ưu bộ nhớ quá sớm khi chưa chắc công thức đúng — hãy viết bản bảng đầy đủ trước.

## 2.3 Áp dụng vào bài này

Chỉ cần **đếm** số BST: chọn gốc `r` thì số cây = (số cây trái với `r−1` nút) × (số cây phải với `n−r` nút) → **DP** `G[n] = Σ G[r−1]·G[n−r]` (số Catalan).

# 3. Quan sát then chốt

Chọn gốc `i`. Cây trái chứa `{1, …, i−1}` (`i − 1` nút), cây phải chứa `{i+1, …, n}` (`n − i` nút).

Số BST trên `{i+1, …, n}` **bằng** số BST trên `{1, …, n−i}` — chỉ cần trừ mọi giá trị đi `i` là ra một song ánh. Vậy số cây chỉ phụ thuộc **số nút**. Đặt `G[k]` = số BST có `k` nút.

Với gốc `i`, mỗi cây trái ghép được với mỗi cây phải → **quy tắc nhân**: `G[i−1] · G[n−i]` cây. Cộng theo mọi gốc:

```text
G[0] = 1                         (cây rỗng)
G[n] = Σ_{i=1..n} G[i−1] · G[n−i]
```

# 4. Baseline: đệ quy theo khoảng (không dùng quan sát trên)

Đếm trực tiếp theo khoảng `[lo, hi]` như cấu trúc của #95, nhưng không memo.

In [6]:
from typing import List
from functools import lru_cache
import math

def count_naive(n: int):
    calls = 0
    def count(lo, hi):
        nonlocal calls
        calls += 1
        if lo >= hi:
            return 1
        return sum(count(lo, i - 1) * count(i + 1, hi) for i in range(lo, hi + 1))
    return count(1, n), calls

for n in [3, 8, 12]:
    print(n, count_naive(n))

3 (5, 15)
8 (1430, 3645)
12 (208012, 295245)


Số lời gọi tăng theo hàm mũ. Có hai cách cải thiện: memo theo `(lo, hi)` (`O(n²)` trạng thái), hoặc dùng quan sát "chỉ phụ thuộc số nút" (`O(n)` trạng thái).

# 5. Lời giải chính: DP một chiều

In [7]:
def num_trees_dp(n: int) -> int:
    G = [0] * (n + 1)
    G[0] = 1
    for k in range(1, n + 1):
        for i in range(1, k + 1):
            G[k] += G[i - 1] * G[k - i]
    return G[n]

print([num_trees_dp(n) for n in range(1, 11)])

[1, 2, 5, 14, 42, 132, 429, 1430, 4862, 16796]


# 6. Mô phỏng: điền bảng `G`

In [8]:
def trace_G(n: int) -> int:
    G = [1] + [0] * n
    print("G[0] = 1")
    for k in range(1, n + 1):
        terms = [f"G[{i-1}]·G[{k-i}]" for i in range(1, k + 1)]
        vals = [G[i - 1] * G[k - i] for i in range(1, k + 1)]
        G[k] = sum(vals)
        print(f"G[{k}] = " + " + ".join(terms) + " = " + " + ".join(map(str, vals)) + f" = {G[k]}")
    return G[n]

_ = trace_G(5)

G[0] = 1
G[1] = G[0]·G[0] = 1 = 1
G[2] = G[0]·G[1] + G[1]·G[0] = 1 + 1 = 2
G[3] = G[0]·G[2] + G[1]·G[1] + G[2]·G[0] = 2 + 1 + 2 = 5
G[4] = G[0]·G[3] + G[1]·G[2] + G[2]·G[1] + G[3]·G[0] = 5 + 2 + 2 + 5 = 14
G[5] = G[0]·G[4] + G[1]·G[3] + G[2]·G[2] + G[3]·G[1] + G[4]·G[0] = 14 + 5 + 4 + 5 + 14 = 42


Chú ý tính **đối xứng**: `G[i−1] · G[k−i]` và `G[k−i] · G[i−1]` (gốc `i` và gốc `k + 1 − i`) bằng nhau — vì phản chiếu cây qua trục dọc (đổi trái/phải, đổi `v` thành `k + 1 − v`) là một song ánh.

# 7. Công thức đóng: số Catalan

`G[n]` chính là **số Catalan** `C_n`:

```text
C_n = (2n)! / ((n+1)! · n!) = C(2n, n) / (n + 1)
```

Hoặc dùng truy hồi một bước (từ công thức trên): `C_{n+1} = C_n · 2(2n + 1) / (n + 2)` — chỉ `O(n)` phép tính.

In [9]:
def num_trees_formula(n: int) -> int:
    return math.comb(2 * n, n) // (n + 1)

def num_trees_iter(n: int) -> int:
    c = 1
    for k in range(n):
        c = c * 2 * (2 * k + 1) // (k + 2)
    return c

print([num_trees_formula(n) for n in range(1, 11)])
print([num_trees_iter(n) for n in range(1, 11)])

[1, 2, 5, 14, 42, 132, 429, 1430, 4862, 16796]
[1, 2, 5, 14, 42, 132, 429, 1430, 4862, 16796]


Phép chia `//` trong `num_trees_iter` luôn **chia hết** ở mỗi bước (vì mỗi giá trị trung gian chính là một số Catalan). Nếu đổi thứ tự thành `c * 2 * (2k+1) // (k+2)` thì đúng; còn `c // (k+2) * 2 * (2k+1)` sẽ **sai** vì chia trước có thể không chia hết.

# 8. Phiên bản đúng format LeetCode

In [10]:
class Solution:
    def numTrees(self, n: int) -> int:
        G = [1] + [0] * n
        for k in range(1, n + 1):
            for i in range(1, k + 1):
                G[k] += G[i - 1] * G[k - i]
        return G[n]

In [11]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<10} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.numTrees(3), 5)
check("Ví dụ 2", sol.numTrees(1), 1)
check("n = 19", sol.numTrees(19), 1767263190)

for n in range(1, 13):
    assert sol.numTrees(n) == count_naive(n)[0]
for n in range(1, 200):
    assert sol.numTrees(n) == num_trees_formula(n) == num_trees_iter(n)
print("n <= 12 khớp đệ quy; n < 200 khớp công thức Catalan")
print("2^31 - 1 =", 2**31 - 1, "| C_19 =", sol.numTrees(19), "| C_20 =", sol.numTrees(20))

Ví dụ 1    PASS  5
Ví dụ 2    PASS  1
n = 19     PASS  1767263190


n <= 12 khớp đệ quy; n < 200 khớp công thức Catalan
2^31 - 1 = 2147483647 | C_19 = 1767263190 | C_20 = 6564120420


`C_19 = 1 767 263 190 < 2^31 − 1` nhưng `C_20` thì vượt — đó là lý do đề giới hạn `n <= 19` (để vừa số nguyên 32-bit trong các ngôn ngữ như Java/C++).

# 9. Vì sao đúng? Độ phức tạp?

- Mỗi BST có đúng một gốc → các trường hợp "gốc = i" **rời nhau** → cộng.
- Với gốc cố định, cây trái và cây phải chọn **độc lập** → nhân.
- Số BST chỉ phụ thuộc số nút (song ánh dịch giá trị).

```text
DP:            Time O(n²),  Space O(n)
Công thức:     Time O(n) (hoặc O(n) phép nhân số lớn), Space O(1)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: `G[0] = 0`

Cây rỗng là **một** cấu hình hợp lệ (cho phía trái/phải của gốc). Nếu `G[0] = 0`, mọi tích đều bằng 0.

In [12]:
def wrong_G0(n: int) -> int:
    G = [0] * (n + 1)                              # SAI: G[0] phải là 1
    G[1] = 1
    for k in range(2, n + 1):
        for i in range(1, k + 1):
            G[k] += G[i - 1] * G[k - i]
    return G[n]

print("Sai :", [wrong_G0(n) for n in range(1, 6)], "| Đúng:", [sol.numTrees(n) for n in range(1, 6)])

Sai : [1, 0, 1, 0, 2] | Đúng: [1, 2, 5, 14, 42]


## Lỗi 2: Cộng thay vì nhân

In [13]:
def wrong_add(n: int) -> int:
    G = [1] + [0] * n
    for k in range(1, n + 1):
        for i in range(1, k + 1):
            G[k] += G[i - 1] + G[k - i]            # SAI: phải nhân
    return G[n]

print("Sai :", [wrong_add(n) for n in range(1, 6)], "| Đúng:", [sol.numTrees(n) for n in range(1, 6)])

Sai : [2, 6, 18, 54, 162] | Đúng: [1, 2, 5, 14, 42]


Mỗi cây trái ghép với **mỗi** cây phải → số cặp là tích, không phải tổng.

## Lỗi 3: Chia trước khi nhân trong công thức lặp

In [14]:
def wrong_divide_first(n: int) -> int:
    c = 1
    for k in range(n):
        c = c // (k + 2) * 2 * (2 * k + 1)         # SAI: chia trước, mất phần dư
    return c

print("Sai :", [wrong_divide_first(n) for n in range(1, 6)], "| Đúng:", [sol.numTrees(n) for n in range(1, 6)])

Sai : [0, 0, 0, 0, 0] | Đúng: [1, 2, 5, 14, 42]


## Lỗi 4: Dùng số thực

`math.factorial(2n) / (...)` cho kết quả `float` — với `n` lớn sẽ mất chính xác. Luôn dùng `//` với số nguyên (hoặc `math.comb`).

In [15]:
n = 40
print("float :", math.factorial(2 * n) / (math.factorial(n + 1) * math.factorial(n)))
print("int   :", math.factorial(2 * n) // (math.factorial(n + 1) * math.factorial(n)))

float : 2.6221270422764923e+21
int   : 2622127042276492108820


# 11. Mẹo nhận diện pattern

Truy hồi `X[n] = Σ X[i] · X[n−1−i]` (chia thành **hai phần độc lập** quanh một "điểm chọn") → **số Catalan**. Các bài ra số Catalan:

| Bài | "Điểm chọn" |
|---|---|
| #96 Unique BSTs | gốc |
| #22 Generate Parentheses | cặp ngoặc chứa ký tự đầu |
| #894 All Possible Full Binary Trees | phân chia nút cho hai con |
| Số cách tam giác phân đa giác lồi `n + 2` cạnh | tam giác chứa một cạnh cố định |
| #1259 Handshakes That Don't Cross | người bắt tay với người 1 |
| Số dãy stack hợp lệ (push/pop) | lần pop phần tử đầu tiên |

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Tính `G[6]` từ `G[0..5] = 1, 1, 2, 5, 14, 42`.

## Bài 2 — Điền chỗ trống

In [16]:
def exercise_2_num_trees(n: int) -> int:
    G = [1] + [0] * n
    for k in range(1, n + 1):
        # TODO: G[k] = tổng theo mọi gốc i
        pass
    return G[n]


# assert exercise_2_num_trees(3) == 5
# assert exercise_2_num_trees(1) == 1
# assert exercise_2_num_trees(19) == 1767263190

## Bài 3 — Biến thể: đếm cách đặt ngoặc đúng (liên hệ #22)

Đếm số chuỗi ngoặc đúng có `n` cặp, bằng DP **theo cấu trúc ngoặc** (không dùng công thức Catalan): chuỗi đúng không rỗng có dạng `(A)B` với `A`, `B` đúng.

In [17]:
def exercise_3_count_parentheses(n: int) -> int:
    # TODO
    pass


# assert exercise_3_count_parentheses(3) == 5
# assert exercise_3_count_parentheses(0) == 1
# assert [exercise_3_count_parentheses(k) for k in range(8)] == [1, 1, 2, 5, 14, 42, 132, 429]

## Bài 4 — Biến thể: số BST có chiều cao đúng bằng `h`

Đếm số BST với `n` nút có **chiều cao** (số nút trên đường dài nhất từ gốc tới lá) **không vượt quá** `h`, rồi suy ra số cây có chiều cao **đúng bằng** `h`.

```python
count_height_exact(3, 2) -> 1     # chỉ cây cân bằng 2 làm gốc
count_height_exact(3, 3) -> 4
```

Gợi ý nhanh: `A[k][h]` = số BST `k` nút cao `<= h`; `A[0][h] = 1`; `A[k][h] = Σ A[i−1][h−1] · A[k−i][h−1]`. Chính xác bằng `h` = `A[n][h] − A[n][h−1]`.

In [18]:
def exercise_4_count_height_exact(n: int, h: int) -> int:
    # TODO
    pass


# assert exercise_4_count_height_exact(3, 2) == 1
# assert exercise_4_count_height_exact(3, 3) == 4
# assert sum(exercise_4_count_height_exact(5, h) for h in range(1, 6)) == 42

## Bài 5 — Mini challenge: Catalan modulo `10^9 + 7` cho `n` lớn

Tính `C_n mod (10^9 + 7)` với `n` tới `10^6` trong `O(n)`, dùng **nghịch đảo modulo** (định lý Fermat nhỏ: `a^(p−2) ≡ a^(-1) mod p` với `p` nguyên tố).

In [19]:
def exercise_5_catalan_mod(n: int) -> int:
    # TODO
    pass


# assert exercise_5_catalan_mod(19) == 1767263190 % (10**9 + 7)
# assert all(exercise_5_catalan_mod(k) == (math.comb(2 * k, k) // (k + 1)) % (10**9 + 7) for k in range(0, 300))

# 13. Gợi ý

**Bài 1**
- Mức 1: `G[6] = 1·42 + 1·14 + 2·5 + 5·2 + 14·1 + 42·1`.

**Bài 3**
- Mức 1: `P[0] = 1`; `P[n] = Σ_{a=0..n−1} P[a] · P[n−1−a]` (cặp ngoặc đầu bọc `a` cặp).

**Bài 4**
- Mức 1: bảng `A` kích thước `(n+1) × (h+1)`; `A[k][0] = 1` nếu `k == 0` ngược lại 0.

**Bài 5**
- Mức 1: tính `(2n)!` và `n!`, `(n+1)!` modulo `p`.
- Mức 2: `pow(x, p − 2, p)` là nghịch đảo của `x`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [20]:
_ = trace_G(6)

G[0] = 1
G[1] = G[0]·G[0] = 1 = 1
G[2] = G[0]·G[1] + G[1]·G[0] = 1 + 1 = 2
G[3] = G[0]·G[2] + G[1]·G[1] + G[2]·G[0] = 2 + 1 + 2 = 5
G[4] = G[0]·G[3] + G[1]·G[2] + G[2]·G[1] + G[3]·G[0] = 5 + 2 + 2 + 5 = 14
G[5] = G[0]·G[4] + G[1]·G[3] + G[2]·G[2] + G[3]·G[1] + G[4]·G[0] = 14 + 5 + 4 + 5 + 14 = 42
G[6] = G[0]·G[5] + G[1]·G[4] + G[2]·G[3] + G[3]·G[2] + G[4]·G[1] + G[5]·G[0] = 42 + 14 + 10 + 10 + 14 + 42 = 132


## Đáp án Bài 2

In [21]:
def answer_2_num_trees(n: int) -> int:
    return num_trees_dp(n)

assert answer_2_num_trees(3) == 5
assert answer_2_num_trees(1) == 1
assert answer_2_num_trees(19) == 1767263190
print("Bài 2 PASS (giống num_trees_dp ở mục 5)")

Bài 2 PASS (giống num_trees_dp ở mục 5)


## Đáp án Bài 3

In [22]:
def answer_3_count_parentheses(n: int) -> int:
    P = [1] + [0] * n
    for k in range(1, n + 1):
        P[k] = sum(P[a] * P[k - 1 - a] for a in range(k))
    return P[n]

assert answer_3_count_parentheses(3) == 5
assert answer_3_count_parentheses(0) == 1
assert [answer_3_count_parentheses(k) for k in range(8)] == [1, 1, 2, 5, 14, 42, 132, 429]

def gen_parens(n):
    if n == 0:
        return [""]
    return ["(" + A + ")" + B for a in range(n) for A in gen_parens(a) for B in gen_parens(n - 1 - a)]

assert all(len(gen_parens(k)) == answer_3_count_parentheses(k) for k in range(8))
print("Bài 3 PASS | n = 3:", gen_parens(3))

Bài 3 PASS | n = 3: ['()()()', '()(())', '(())()', '(()())', '((()))']


## Đáp án Bài 4

In [23]:
def answer_4_count_height_exact(n: int, h: int) -> int:
    def at_most(H):
        A = [[0] * (H + 1) for _ in range(n + 1)]
        for hh in range(H + 1):
            A[0][hh] = 1
        for hh in range(1, H + 1):
            for k in range(1, n + 1):
                A[k][hh] = sum(A[i - 1][hh - 1] * A[k - i][hh - 1] for i in range(1, k + 1))
        return A[n][H]
    return at_most(h) - (at_most(h - 1) if h >= 1 else 0)

assert answer_4_count_height_exact(3, 2) == 1
assert answer_4_count_height_exact(3, 3) == 4
assert sum(answer_4_count_height_exact(5, h) for h in range(1, 6)) == 42
print("Bài 4 PASS | phân bố chiều cao cho n = 7:", {h: answer_4_count_height_exact(7, h) for h in range(1, 8)})

Bài 4 PASS | phân bố chiều cao cho n = 7: {1: 0, 2: 0, 3: 1, 4: 68, 5: 152, 6: 144, 7: 64}


## Đáp án Bài 5

In [24]:
def answer_5_catalan_mod(n: int) -> int:
    p = 10**9 + 7
    fact = [1] * (2 * n + 2)
    for i in range(1, 2 * n + 2):
        fact[i] = fact[i - 1] * i % p
    return fact[2 * n] * pow(fact[n + 1], p - 2, p) % p * pow(fact[n], p - 2, p) % p

assert answer_5_catalan_mod(19) == 1767263190 % (10**9 + 7)
assert all(answer_5_catalan_mod(k) == (math.comb(2 * k, k) // (k + 1)) % (10**9 + 7) for k in range(0, 300))
print("Bài 5 PASS | C_1000000 mod p =", answer_5_catalan_mod(10**6))

Bài 5 PASS | C_1000000 mod p = 70646122


# 15. Tổng kết

> Số BST chỉ phụ thuộc số nút. `G[0] = 1`, `G[n] = Σ_{i=1..n} G[i−1] · G[n−i]` (quy tắc cộng theo gốc, quy tắc nhân cho hai phía). Đó là số Catalan `C(2n, n) / (n + 1)`.

---

## Checklist trước khi nộp

- [ ] `G[0] = 1`?
- [ ] Nhân `G[i−1] · G[n−i]`, không cộng?
- [ ] Nếu dùng công thức: số nguyên, nhân trước chia sau?
- [ ] Nhận ra được dạng Catalan ở bài khác?